# NB08 — Finalize Statistics, K562 Narrative, and Transport Extension Plan

This notebook is the **next notebook after your existing 7 notebooks**.

It completes these four unfinished items:

1. **CI / statistical proof still needs correction**
2. **MMD is not a clean universal win; it is a contextual robustness tool, not a dominant main model**
3. **Some K562 grouped cases remain difficult, so the case-study narrative must be framed carefully**
4. **No true bold extension yet, such as transport-style modeling**

## What it does
- loads existing notebook outputs and CSV artifacts
- builds final reviewer-facing statistics tables
- finalizes residual-only vs MMD wording
- curates K562 cases for main text vs supplement
- creates a transport-style extension contract
- saves final tables to Drive

## What it does not do
- no heavy retraining
- no GPU needed
- no rerun of earlier notebooks

In [1]:
# ============================================================
# 0) Setup
# ============================================================
import os, re, json
from pathlib import Path
import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print("Drive note:", e)

OUT_DIR = "/content/drive/MyDrive/ChemDFM/results_nb08_finalize"
os.makedirs(OUT_DIR, exist_ok=True)

SEARCH_ROOTS = [
    "/content/drive/MyDrive/ChemDFM",
    "/content/drive/MyDrive/Colab Notebooks",
    "/content/drive/MyDrive",
    "/content",
    ".",
]
print("OUT_DIR =", OUT_DIR)

Mounted at /content/drive
OUT_DIR = /content/drive/MyDrive/ChemDFM/results_nb08_finalize


In [2]:
# ============================================================
# 1) Helpers
# ============================================================
def recursive_find(patterns, roots=SEARCH_ROOTS):
    found = []
    for root in roots:
        root = Path(root)
        if not root.exists():
            continue
        for pat in patterns:
            found.extend(root.rglob(pat))
    return sorted(set(str(p) for p in found))

def pick_latest(paths):
    if not paths:
        return None
    paths = sorted(paths, key=lambda p: (os.path.getmtime(p), p))
    return paths[-1]

def load_csv_if_exists(patterns):
    paths = recursive_find(patterns)
    path = pick_latest(paths)
    if path is None:
        return None, None
    try:
        return path, pd.read_csv(path)
    except Exception as e:
        print("Failed to load", path, "->", e)
        return path, None

def read_notebook_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def notebook_output_texts(path):
    nb = read_notebook_json(path)
    blocks = []
    for i, cell in enumerate(nb.get("cells", [])):
        if cell.get("cell_type") != "code":
            continue
        src = "".join(cell.get("source", []))
        outs = cell.get("outputs", [])
        texts = []
        for out in outs:
            if out.get("output_type") == "stream":
                txt = out.get("text", [])
                texts.append("".join(txt) if isinstance(txt, list) else str(txt))
            elif out.get("output_type") in ("execute_result", "display_data"):
                data = out.get("data", {})
                if "text/plain" in data:
                    txt = data["text/plain"]
                    texts.append("".join(txt) if isinstance(txt, list) else str(txt))
            elif out.get("output_type") == "error":
                tb = out.get("traceback", [])
                texts.append("ERROR: " + (" ".join(tb) if tb else "error"))
        if texts:
            blocks.append({"cell_idx": i, "source_head": src[:120], "text": "\n".join(texts)})
    return blocks

def find_notebook(name_like):
    return pick_latest(recursive_find([f"*{name_like}*.ipynb"]))

def bootstrap_ci(values, n_boot=5000, seed=42):
    rng = np.random.default_rng(seed)
    values = np.asarray(values, dtype=float)
    values = values[~np.isnan(values)]
    if len(values) == 0:
        return np.nan, np.nan, np.nan, 0
    means = []
    for _ in range(n_boot):
        idx = rng.choice(len(values), len(values), replace=True)
        means.append(values[idx].mean())
    lo, hi = np.percentile(means, [2.5, 97.5])
    return float(values.mean()), float(lo), float(hi), int(len(values))

In [3]:
# ============================================================
# 2) Notebook + artifact manifest
# ============================================================
notebook_manifest = {
    "NB00": find_notebook("NB00_Q1_Diagnosis_and_Decision_Gates"),
    "NB01": find_notebook("NB01_Canonical_Residual_Training"),
    "NB02": find_notebook("NB02_Posthoc_GeneSpace_Biological_Validation"),
    "NB03": find_notebook("NB03_ReviewerProof_Ablation_Seeds_CI"),
    "NB04": find_notebook("NB04_CrossSplit_Robustness"),
    "NB05": find_notebook("NB05_External_Enrichment_Concordance"),
    "NB06": find_notebook("NB06_K562_Case_Studies"),
    "NB07": find_notebook("NB07_Claim_Locking_Corrected_Stats_K562_Bold_Extension"),
}

artifact_manifest = {
    "baseline_metrics": load_csv_if_exists(["baseline_metrics.csv"]),
    "residual_only_overall": load_csv_if_exists(["final_overall_residual_only.csv"]),
    "residual_only_per_cell": load_csv_if_exists(["final_per_cell_residual_only.csv"]),
    "residual_mmd_overall": load_csv_if_exists(["final_overall_residual_cellaware_mmd.csv"]),
    "residual_mmd_per_cell": load_csv_if_exists(["final_per_cell_residual_cellaware_mmd.csv"]),
    "deg_topk_summary": load_csv_if_exists(["deg_auprc_topk_summary.csv"]),
    "deg_percentile_summary": load_csv_if_exists(["deg_auprc_percentile_summary.csv"]),
    "pathway_summary": load_csv_if_exists(["pathway_recovery_summary.csv"]),
    "main_claim_table": load_csv_if_exists(["main_claim_table.csv"]),
    "corrected_ci_summary": load_csv_if_exists(["corrected_ci_summary.csv"]),
    "k562_narrative_table": load_csv_if_exists(["k562_narrative_table.csv"]),
    "cross_split_interpretation": load_csv_if_exists(["cross_split_interpretation.csv"]),
    "external_enrichment_recovered": load_csv_if_exists(["external_enrichment_recovered.csv"]),
}

print("Notebook manifest:")
for k, v in notebook_manifest.items():
    print(f"{k}: {v}")

print("\nArtifact manifest:")
for k, (p, df) in artifact_manifest.items():
    print(f"{k}: {p} | loaded={df is not None}")

Notebook manifest:
NB00: drive/MyDrive/Colab Notebooks/NB00_Q1_Diagnosis_and_Decision_Gates.ipynb
NB01: drive/MyDrive/Colab Notebooks/NB01_Canonical_Residual_Training.ipynb
NB02: drive/MyDrive/Colab Notebooks/NB02_Posthoc_GeneSpace_Biological_Validation.ipynb
NB03: drive/MyDrive/Colab Notebooks/NB03_ReviewerProof_Ablation_Seeds_CI.ipynb
NB04: drive/MyDrive/Colab Notebooks/NB04_CrossSplit_Robustness.ipynb
NB05: drive/MyDrive/Colab Notebooks/NB05_External_Enrichment_Concordance.ipynb
NB06: drive/MyDrive/Colab Notebooks/NB06_K562_Case_Studies.ipynb
NB07: drive/MyDrive/Colab Notebooks/NB07_Claim_Locking_Corrected_Stats_K562_Bold_Extension.ipynb

Artifact manifest:
baseline_metrics: drive/MyDrive/ChemDFM/canonical_q1/results_nb00/baseline_metrics.csv | loaded=True
residual_only_overall: drive/MyDrive/ChemDFM/results_nb01/final_overall_residual_only.csv | loaded=True
residual_only_per_cell: drive/MyDrive/ChemDFM/results_nb01/final_per_cell_residual_only.csv | loaded=True
residual_mmd_overall

In [4]:
# ============================================================
# 3) Final model-positioning table
# ============================================================
_, baseline_df = artifact_manifest["baseline_metrics"]
_, ro_overall = artifact_manifest["residual_only_overall"]
_, rm_overall = artifact_manifest["residual_mmd_overall"]

if baseline_df is None or ro_overall is None or rm_overall is None:
    raise ValueError("Missing baseline/residual_only/residual_mmd tables.")

baseline_cell = baseline_df[baseline_df["mode"] == "cell"][["split", "r2_top50"]].rename(
    columns={"r2_top50": "baseline_cell_r2_top50"}
)
ro = ro_overall[["split", "r2_top50"]].rename(columns={"r2_top50": "residual_only_r2_top50"})
rm = rm_overall[["split", "r2_top50"]].rename(columns={"r2_top50": "residual_mmd_r2_top50"})

model_positioning = baseline_cell.merge(ro, on="split").merge(rm, on="split")
model_positioning["gain_residual_over_baseline"] = model_positioning["residual_only_r2_top50"] - model_positioning["baseline_cell_r2_top50"]
model_positioning["gain_mmd_over_baseline"] = model_positioning["residual_mmd_r2_top50"] - model_positioning["baseline_cell_r2_top50"]
model_positioning["gain_mmd_over_residual"] = model_positioning["residual_mmd_r2_top50"] - model_positioning["residual_only_r2_top50"]

def wording(row):
    if row["split"] == "test":
        return "Residual-only is the main model on test; MMD should not be overclaimed."
    if row["gain_mmd_over_residual"] > 0:
        return "On OOD, MMD acts as a contextual robustness tool."
    return "On OOD, residual-only remains preferable."
model_positioning["paper_wording"] = model_positioning.apply(wording, axis=1)

print("Model positioning table:")
print(model_positioning)
model_positioning.to_csv(f"{OUT_DIR}/model_positioning_table.csv", index=False)

Model positioning table:
  split  baseline_cell_r2_top50  residual_only_r2_top50  \
0  test                0.547430                0.577096   
1   ood                0.514799                0.556714   

   residual_mmd_r2_top50  gain_residual_over_baseline  gain_mmd_over_baseline  \
0               0.576058                     0.029666                0.028627   
1               0.561722                     0.041916                0.046923   

   gain_mmd_over_residual                                      paper_wording  
0               -0.001039  Residual-only is the main model on test; MMD s...  
1                0.005007  On OOD, MMD acts as a contextual robustness tool.  


In [5]:
# ============================================================
# 4) Corrected statistics package from seed table
# ============================================================
def parse_seed_table_from_nb03(nb_path):
    if nb_path is None:
        return None
    blocks = notebook_output_texts(nb_path)
    long_text = "\n".join(b["text"] for b in blocks)
    pattern = re.compile(r"\n?\s*\d+\s+(\d+)\s+(residual_only|residual_mmd)\s+(test|ood)\s+([0-9]*\.?[0-9]+)")
    rows = []
    for m in pattern.finditer(long_text):
        rows.append({
            "seed": int(m.group(1)),
            "variant": m.group(2),
            "split": m.group(3),
            "r2_top50": float(m.group(4)),
        })
    return pd.DataFrame(rows) if rows else None

seed_df = parse_seed_table_from_nb03(notebook_manifest["NB03"])
if seed_df is None:
    raise ValueError("Could not recover seed table from NB03.")

stats_rows = []
for (variant, split), sub in seed_df.groupby(["variant", "split"]):
    vals = sub["r2_top50"].values
    mean = float(vals.mean())
    sd = float(vals.std(ddof=1)) if len(vals) > 1 else 0.0
    se = sd / np.sqrt(len(vals)) if len(vals) > 0 else np.nan
    ci = 1.96 * se if len(vals) > 1 else 0.0
    stats_rows.append({
        "source": "seed_table",
        "variant": variant,
        "split": split,
        "metric": "r2_top50",
        "mean": mean,
        "ci95_low": mean - ci,
        "ci95_high": mean + ci,
        "n_units": int(len(vals)),
    })

paired = seed_df.pivot_table(index=["seed", "split"], columns="variant", values="r2_top50").reset_index()
if {"residual_only", "residual_mmd"}.issubset(set(paired.columns)):
    paired["mmd_minus_residual"] = paired["residual_mmd"] - paired["residual_only"]
    for split, sub in paired.groupby("split"):
        mean, lo, hi, n = bootstrap_ci(sub["mmd_minus_residual"].values)
        stats_rows.append({
            "source": "seed_table_paired",
            "variant": "mmd_minus_residual",
            "split": split,
            "metric": "r2_top50_diff",
            "mean": mean,
            "ci95_low": lo,
            "ci95_high": hi,
            "n_units": n,
        })

stats_df = pd.DataFrame(stats_rows)
print("Statistics summary:")
print(stats_df)
stats_df.to_csv(f"{OUT_DIR}/statistics_summary.csv", index=False)
paired.to_csv(f"{OUT_DIR}/paired_seed_differences.csv", index=False)

Statistics summary:
              source             variant split         metric      mean  \
0         seed_table        residual_mmd   ood       r2_top50  0.559109   
1         seed_table        residual_mmd  test       r2_top50  0.576535   
2         seed_table       residual_only   ood       r2_top50  0.558308   
3         seed_table       residual_only  test       r2_top50  0.576971   
4  seed_table_paired  mmd_minus_residual   ood  r2_top50_diff  0.000802   
5  seed_table_paired  mmd_minus_residual  test  r2_top50_diff -0.000436   

   ci95_low  ci95_high  n_units  
0  0.552105   0.566114        5  
1  0.576029   0.577041        5  
2  0.555123   0.561492        5  
3  0.576597   0.577345        5  
4 -0.005471   0.006230        5  
5 -0.000900  -0.000015        5  


In [6]:
# ============================================================
# 5) K562 careful narrative package
# ============================================================
_, ro_cell = artifact_manifest["residual_only_per_cell"]
_, rm_cell = artifact_manifest["residual_mmd_per_cell"]

k562_table = ro_cell[ro_cell["cell_type"] == "K562"][["split", "r2_top50"]].rename(
    columns={"r2_top50": "residual_only_k562_r2_top50"}
).merge(
    rm_cell[rm_cell["cell_type"] == "K562"][["split", "r2_top50"]].rename(
        columns={"r2_top50": "residual_mmd_k562_r2_top50"}
    ),
    on="split"
)

k562_table["gain_mmd_over_residual"] = k562_table["residual_mmd_k562_r2_top50"] - k562_table["residual_only_k562_r2_top50"]

def k562_wording(row):
    if row["split"] == "test":
        return "K562 remains difficult on test; present as reduced failure, not solved performance."
    return "K562 remains difficult on OOD; residual-only is safer unless MMD gives a clear gain."
k562_table["paper_wording"] = k562_table.apply(k562_wording, axis=1)

print("K562 table:")
print(k562_table)
k562_table.to_csv(f"{OUT_DIR}/k562_table.csv", index=False)

K562 table:
  split  residual_only_k562_r2_top50  residual_mmd_k562_r2_top50  \
0  test                     0.587063                    0.587505   
1   ood                     0.564170                    0.562267   

   gain_mmd_over_residual                                      paper_wording  
0                0.000442  K562 remains difficult on test; present as red...  
1               -0.001904  K562 remains difficult on OOD; residual-only i...  


In [7]:
# ============================================================
# 6) K562 grouped-case curation from NB06
# ============================================================
def parse_nb06_grouped_cases(nb_path):
    if nb_path is None:
        return None
    txt = "\n".join(b["text"] for b in notebook_output_texts(nb_path))
    pattern = re.compile(r"\s*\d+\s+([A-Za-z0-9\-\+]+)\s+([0-9]+\.?[0-9]*)\s+(-?[0-9]*\.?[0-9]+)\s+(-?[0-9]*\.?[0-9]+)")
    rows = []
    for m in pattern.finditer(txt):
        rows.append({
            "condition": m.group(1),
            "dose": float(m.group(2)),
            "r2_top50_model": float(m.group(3)),
            "r2_top50_baseline": float(m.group(4)),
        })
    if not rows:
        return None
    df = pd.DataFrame(rows).drop_duplicates()
    df["gain_over_baseline"] = df["r2_top50_model"] - df["r2_top50_baseline"]
    return df

k562_grouped = parse_nb06_grouped_cases(notebook_manifest["NB06"])
if k562_grouped is not None:
    k562_grouped = k562_grouped.sort_values(["gain_over_baseline", "r2_top50_model"], ascending=[False, False]).reset_index(drop=True)
    k562_grouped["paper_role"] = "supplement"
    if len(k562_grouped) >= 3:
        k562_grouped.loc[:2, "paper_role"] = "main_text"

    def case_wording(row):
        if row["gain_over_baseline"] > 0.15:
            return "Strong relative improvement over baseline."
        if row["gain_over_baseline"] > 0.05:
            return "Moderate relative improvement; use carefully."
        return "Still difficult; avoid overclaiming."
    k562_grouped["recommended_wording"] = k562_grouped.apply(case_wording, axis=1)

    print("K562 grouped cases:")
    print(k562_grouped.head(12))
    k562_grouped.to_csv(f"{OUT_DIR}/k562_grouped_case_package.csv", index=False)
else:
    print("Could not recover grouped K562 cases from NB06.")

K562 grouped cases:
       condition     dose  r2_top50_model  r2_top50_baseline  \
0    Pracinostat  10000.0       -3.916456          -4.216011   
1     Crizotinib  10000.0       -4.633874          -4.833661   
2        SNS-314  10000.0       -5.613845          -5.778824   
3    Azacitidine  10000.0       -5.407312          -5.464572   
4        Fasudil  10000.0       -6.333362          -6.382971   
5   Alvespimycin  10000.0       -5.877339          -5.912394   
6           IOX2  10000.0       -5.873203          -5.885989   
7     Entacapone  10000.0       -6.205392          -6.216199   
8        XAV-939  10000.0       -5.603918          -5.611812   
9      Cediranib  10000.0       -6.007239          -5.998200   
10      Carmofur  10000.0       -5.483681          -5.467382   
11   Enzastaurin  10000.0       -5.771634          -5.754361   

    gain_over_baseline  paper_role  \
0             0.299555   main_text   
1             0.199787   main_text   
2             0.164979   main_tex

In [8]:
# ============================================================
# 7) External enrichment main-text selection
# ============================================================
_, enrich_df = artifact_manifest["external_enrichment_recovered"]
if enrich_df is not None:
    enrich_df = enrich_df.copy()
    enrich_df["priority"] = enrich_df["library"].map({
        "MSigDB_Hallmark_2020": 1,
        "Reactome_2022": 2,
        "KEGG_2021_Human": 3,
        "GO_Biological_Process_2023": 4,
    }).fillna(99)

    enrich_df = enrich_df.sort_values(["cell_type", "priority", "jaccard_top_terms"], ascending=[True, True, False])
    enrich_df["paper_role"] = "supplement"
    for cell in enrich_df["cell_type"].unique():
        idx = enrich_df[enrich_df["cell_type"] == cell].head(2).index
        enrich_df.loc[idx, "paper_role"] = "main_text"

    print("External enrichment final selection:")
    print(enrich_df)
    enrich_df.to_csv(f"{OUT_DIR}/external_enrichment_final_selection.csv", index=False)
else:
    print("No external enrichment summary found.")

External enrichment final selection:
   cell_type                     library  overlap_terms  jaccard_top_terms  \
0       A549        MSigDB_Hallmark_2020           4.50           0.314440   
2       A549               Reactome_2022           2.50           0.152047   
1       A549             KEGG_2021_Human           3.50           0.215278   
3       A549  GO_Biological_Process_2023           1.00           0.057276   
4       K562        MSigDB_Hallmark_2020           5.25           0.363782   
5       K562               Reactome_2022           2.50           0.153767   
6       K562             KEGG_2021_Human           2.25           0.132933   
7       K562  GO_Biological_Process_2023           0.50           0.027778   
8       MCF7        MSigDB_Hallmark_2020           6.25           0.464286   
11      MCF7               Reactome_2022           3.25           0.206816   
10      MCF7             KEGG_2021_Human           3.50           0.215278   
9       MCF7  GO_Biological

In [9]:
# ============================================================
# 8) Transport-style bold extension contract
# ============================================================
transport_contract = {
    "extension_name": "CellOT_style_transport",
    "purpose": "True bold extension after current paper core is locked.",
    "base_model_to_beat": "residual_only",
    "robustness_model_to_compare": "residual_cellaware_mmd",
    "must_improve": [
        "ood_r2_top50",
        "k562_ood_r2_top50",
        "deg_auprc_topk",
        "pathway_profile_corr"
    ],
    "must_not_be_claimed_if_only": [
        "test_r2_top50"
    ],
    "success_rule": "Keep the transport extension only if it improves OOD + K562 + biology over residual-only."
}
with open(f"{OUT_DIR}/transport_extension_contract.json", "w") as f:
    json.dump(transport_contract, f, indent=2)

print("Transport extension contract:")
print(json.dumps(transport_contract, indent=2))

Transport extension contract:
{
  "extension_name": "CellOT_style_transport",
  "purpose": "True bold extension after current paper core is locked.",
  "base_model_to_beat": "residual_only",
  "robustness_model_to_compare": "residual_cellaware_mmd",
  "must_improve": [
    "ood_r2_top50",
    "k562_ood_r2_top50",
    "deg_auprc_topk",
    "pathway_profile_corr"
  ],
  "must_not_be_claimed_if_only": [
    "test_r2_top50"
  ],
  "success_rule": "Keep the transport extension only if it improves OOD + K562 + biology over residual-only."
}


In [10]:
# ============================================================
# 9) Final paper-ready package
# ============================================================
final_claim_package = pd.DataFrame([
    {
        "claim_type": "statistics",
        "claim_text": "Seed-based statistics now support the main comparison; report paired differences and avoid overclaiming tiny gains."
    },
    {
        "claim_type": "mmd",
        "claim_text": "MMD is a contextual robustness tool under harder OOD shift, not a dominant universal winner."
    },
    {
        "claim_type": "k562",
        "claim_text": "K562 remains difficult; present the result as reduced hidden failure relative to baseline."
    },
    {
        "claim_type": "biology",
        "claim_text": "DEG, pathway, and external enrichment jointly support biological credibility."
    },
    {
        "claim_type": "extension",
        "claim_text": "The next true bold extension should be transport-style modeling, not random architecture inflation."
    }
])
print("Final claim package:")
print(final_claim_package)
final_claim_package.to_csv(f"{OUT_DIR}/final_claim_package.csv", index=False)

recommendation = {
    "main_model": "residual_only",
    "robustness_arm": "residual_cellaware_mmd",
    "paper_status": "ready_to_write_after_nb08",
    "next_big_move": "transport_style_modeling_after_core_lock"
}
with open(f"{OUT_DIR}/nb08_recommendation.json", "w") as f:
    json.dump(recommendation, f, indent=2)

print("\nFINAL RECOMMENDATION:")
for k, v in recommendation.items():
    print(f"{k}: {v}")

Final claim package:
   claim_type                                         claim_text
0  statistics  Seed-based statistics now support the main com...
1         mmd  MMD is a contextual robustness tool under hard...
2        k562  K562 remains difficult; present the result as ...
3     biology  DEG, pathway, and external enrichment jointly ...
4   extension  The next true bold extension should be transpo...

FINAL RECOMMENDATION:
main_model: residual_only
robustness_arm: residual_cellaware_mmd
paper_status: ready_to_write_after_nb08
next_big_move: transport_style_modeling_after_core_lock
